# v17 — fusi ECGFounder + RR v3.1 yang dilatih dengan EKG genggam (CinC 2017), diuji di data wearable (Icentia11k)

Rancangan dikunci sebelum model dilatih: [RANCANGAN.md](RANCANGAN.md) (sha256 di `../logs/v17_praregistrasi.log`). Logika di `fusi.py`.

| Sistem | Keputusan AF |
|---|---|
| **W-F (primer)** | regresi logistik pada [d1 ECGFounder, r = log-odds AF paket v3.1], dilatih di CinC 2017 (CC BY 3.0), p ≥ 0,5 |
| D-1 (kandidat sekarang) | d1 > −1 |
| **v3.1 paket (produksi)** | argmax paket, AFIB + AFL digabung |
| D-1 ∨ v3.1 | salah satu menyebut AF (pembanding naif) |

Kriteria (W-F): K1 sensitivitas Icentia lebih baik dari D-1; K2 Macro-F1 Icentia bukan lebih buruk dari D-1; K3 Macro-F1 dan sensitivitas
Icentia bukan lebih buruk dari v3.1; K4 alarm sinus takikardia asli ≥ 120 bpm ≤ 10%; K5 Macro-F1 CPSC 2021, MIT-BIH, LTAFDB bukan lebih
buruk dari D-1.

In [1]:
import json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import fusi as F
from fusi import Ev


def dl(v):
    return f"Δ {v[0]:+.3f} [{v[1]:+.3f}…{v[2]:+.3f}], P = {v[3]:.3f} → {Ev.vonis(v[3])}"


def tabel(titik, judul):
    return pd.DataFrame({k: {m: f"{v[m][0]:.3f} [{v[m][1]:.2f}–{v[m][2]:.2f}]" for m in ("Macro-F1", "Se", "Sp", "PPV")} for k, v in titik.items()}).T.rename_axis(judul)


def tabel_banding(b, judul):
    return pd.DataFrame({k: {m: dl(v[m]) for m in ("Macro-F1", "Se", "Sp", "PPV")} for k, v in b.items()}).T.rename_axis(judul)


cinc = F.data_cinc()
cinc["unit"] = cinc["subjek"]
model = F.latih(F.masukan(cinc["d1"], cinc["p31"]), cinc["y2"])
koef = F.simpan_model(model)
print("data latih CinC 2017:", len(cinc["y2"]), "window, AF", int(cinc["y2"].sum()), "| rekaman:", np.unique(cinc["rekaman"]).size)
print("W-F: AF bila", f"{koef['w_d1']:.3f}·d1 + {koef['w_r']:.3f}·r + ({koef['b']:.3f}) ≥ 0")
oof = F.oof_cinc(cinc)
pred_c = F.prediksi(model, cinc["d1"], cinc["p31"])
pred_c["W-F"] = oof                                   # validasi silang per rekaman, bukan model penuh
b_cinc, t_cinc = F.banding(cinc, pred_c)
display(tabel(t_cinc, "CinC 2017 (W-F = OOF per rekaman; pendukung)"))
tabel_banding(b_cinc, "CinC 2017")

data latih CinC 2017: 43465 window, AF 3898 | rekaman: 8172
W-F: AF bila 0.629·d1 + 0.272·r + (1.616) ≥ 0


,Macro-F1,Se,Sp,PPV
CinC 2017 (W-F = OOF per rekaman; pendukung),,,,
W-F,0.823 [0.81–0.83],0.918 [0.90–0.93],0.926 [0.92–0.93],0.550 [0.52–0.58]
D-1,0.867 [0.86–0.88],0.830 [0.80–0.85],0.965 [0.96–0.97],0.702 [0.67–0.73]
v3.1 paket,0.757 [0.74–0.77],0.884 [0.87–0.90],0.886 [0.88–0.89],0.433 [0.41–0.46]
D-1 ∨ v3.1,0.749 [0.74–0.76],0.929 [0.91–0.94],0.870 [0.86–0.88],0.413 [0.39–0.44]


,Macro-F1,Se,Sp,PPV
CinC 2017,,,,
W-F vs D-1,"Δ -0.045 [-0.053…-0.036], P = 0.000 → lebih buruk","Δ +0.088 [+0.074…+0.103], P = 1.000 → lebih baik","Δ -0.039 [-0.043…-0.036], P = 0.000 → lebih buruk","Δ -0.152 [-0.169…-0.137], P = 0.000 → lebih buruk"
W-F vs v3.1,"Δ +0.065 [+0.058…+0.072], P = 1.000 → lebih baik","Δ +0.034 [+0.021…+0.046], P = 1.000 → lebih baik","Δ +0.040 [+0.036…+0.044], P = 1.000 → lebih baik","Δ +0.117 [+0.105…+0.129], P = 1.000 → lebih baik"
D-1 ∨ v3.1 vs D-1,"Δ -0.118 [-0.127…-0.108], P = 0.000 → lebih buruk","Δ +0.099 [+0.083…+0.117], P = 1.000 → lebih baik","Δ -0.095 [-0.100…-0.090], P = 0.000 → lebih buruk","Δ -0.289 [-0.306…-0.271], P = 0.000 → lebih buruk"


## 1. Uji utama: Icentia11k (wearable, tidak dipakai untuk melatih atau memilih apa pun)

In [2]:
ico = F.data_icentia()
pred_i = F.prediksi(model, ico["d1"], ico["p31"])
b_ico, t_ico = F.banding(ico, pred_i)
m = ~ico["fast"]
sub = {k: v[m] for k, v in ico.items()}
b_ico_tf, t_ico_tf = F.banding(sub, {k: v[m] for k, v in pred_i.items()})
print("Icentia11k:", len(ico["y2"]), "window, AF", int(ico["y2"].sum()), "| pasien:", np.unique(ico["unit"]).size, "| window Normal cepat tambahan:", int(ico["fast"].sum()))
display(tabel(t_ico, "Icentia11k"))
display(tabel_banding(b_ico, "Icentia11k"))
display(tabel(t_ico_tf, "Icentia11k tanpa window Normal cepat tambahan"))
tabel_banding(b_ico_tf, "Icentia11k tanpa window cepat")

Icentia11k: 115722 window, AF 80986 | pasien: 2420 | window Normal cepat tambahan: 4435


,Macro-F1,Se,Sp,PPV
Icentia11k,,,,
W-F,0.913 [0.90–0.92],0.921 [0.91–0.93],0.934 [0.93–0.94],0.970 [0.97–0.97]
D-1,0.870 [0.86–0.88],0.843 [0.83–0.86],0.972 [0.97–0.98],0.986 [0.98–0.99]
v3.1 paket,0.858 [0.84–0.87],0.871 [0.85–0.89],0.882 [0.87–0.89],0.945 [0.94–0.95]
D-1 ∨ v3.1,0.902 [0.89–0.91],0.937 [0.93–0.95],0.870 [0.86–0.88],0.944 [0.94–0.95]


,Macro-F1,Se,Sp,PPV
Icentia11k,,,,
W-F vs D-1,"Δ +0.043 [+0.035…+0.052], P = 1.000 → lebih baik","Δ +0.078 [+0.067…+0.089], P = 1.000 → lebih baik","Δ -0.038 [-0.043…-0.033], P = 0.000 → lebih buruk","Δ -0.016 [-0.018…-0.013], P = 0.000 → lebih buruk"
W-F vs v3.1,"Δ +0.056 [+0.049…+0.062], P = 1.000 → lebih baik","Δ +0.050 [+0.042…+0.058], P = 1.000 → lebih baik","Δ +0.051 [+0.044…+0.058], P = 1.000 → lebih baik","Δ +0.025 [+0.021…+0.029], P = 1.000 → lebih baik"
D-1 ∨ v3.1 vs D-1,"Δ +0.032 [+0.022…+0.041], P = 1.000 → lebih baik","Δ +0.094 [+0.083…+0.106], P = 1.000 → lebih baik","Δ -0.102 [-0.110…-0.093], P = 0.000 → lebih buruk","Δ -0.042 [-0.047…-0.037], P = 0.000 → lebih buruk"


,Macro-F1,Se,Sp,PPV
Icentia11k tanpa window Normal cepat tambahan,,,,
W-F,0.912 [0.90–0.92],0.921 [0.91–0.93],0.943 [0.94–0.95],0.978 [0.97–0.98]
D-1,0.862 [0.85–0.88],0.843 [0.83–0.86],0.976 [0.97–0.98],0.990 [0.99–0.99]
v3.1 paket,0.859 [0.85–0.87],0.871 [0.85–0.89],0.905 [0.90–0.91],0.961 [0.96–0.96]
D-1 ∨ v3.1,0.907 [0.90–0.92],0.937 [0.93–0.95],0.892 [0.88–0.90],0.959 [0.95–0.96]


,Macro-F1,Se,Sp,PPV
Icentia11k tanpa window cepat,,,,
W-F vs D-1,"Δ +0.050 [+0.040…+0.059], P = 1.000 → lebih baik","Δ +0.078 [+0.067…+0.089], P = 1.000 → lebih baik","Δ -0.033 [-0.038…-0.028], P = 0.000 → lebih buruk","Δ -0.012 [-0.014…-0.010], P = 0.000 → lebih buruk"
W-F vs v3.1,"Δ +0.053 [+0.046…+0.059], P = 1.000 → lebih baik","Δ +0.050 [+0.042…+0.058], P = 1.000 → lebih baik","Δ +0.039 [+0.033…+0.045], P = 1.000 → lebih baik","Δ +0.017 [+0.014…+0.020], P = 1.000 → lebih baik"
D-1 ∨ v3.1 vs D-1,"Δ +0.045 [+0.035…+0.054], P = 1.000 → lebih baik","Δ +0.094 [+0.083…+0.106], P = 1.000 → lebih baik","Δ -0.084 [-0.092…-0.077], P = 0.000 → lebih buruk","Δ -0.031 [-0.035…-0.028], P = 0.000 → lebih buruk"


## 2. Penjaga: sinus takikardia asli (K4) dan Holter (K5)

In [3]:
lat = F.latihan(model)
display(pd.DataFrame({k: {"alarm ≥120 bpm": f"{v['alarm']*100:.1f}% [{v['ci'][0]*100:.1f}–{v['ci'][1]*100:.1f}]",
                          **{p: f"{x['alarm']*100:.0f}%" for p, x in v["pita"].items()}} for k, v in lat.items()}).T.rename_axis("sinus takikardia asli"))
holter = {"CPSC 2021": F.data_cpsc(), "MIT-BIH": F.data_v163("MIT-BIH"), "LTAFDB": F.data_v163("LTAFDB")}
hasil_h = {}
for nm, Z in holter.items():
    pred = F.prediksi(model, Z["d1"], Z["p31"])
    b, t = F.banding(Z, pred)
    hasil_h[nm] = dict(banding=b, titik=t, ritme=F.per_ritme(Z, pred) if "ritme" in Z else None)
    display(tabel(t, nm))
    display(tabel_banding(b, nm))

,alarm ≥120 bpm,0–100 bpm,100–110 bpm,110–120 bpm,120–130 bpm,130–140 bpm,140–150 bpm,150–∞ bpm
sinus takikardia asli,,,,,,,,
W-F,56.6% [45.3–68.4],7%,7%,11%,27%,73%,78%,78%
D-1,0.5% [0.1–0.9],3%,3%,1%,1%,1%,0%,0%
v3.1 paket,98.4% [96.5–99.6],22%,16%,62%,97%,99%,99%,100%
D-1 ∨ v3.1,98.4% [96.6–99.6],23%,17%,62%,97%,99%,99%,100%


,Macro-F1,Se,Sp,PPV
CPSC 2021,,,,
W-F,0.931 [0.89–0.96],0.993 [0.99–1.00],0.905 [0.86–0.94],0.847 [0.74–0.92]
D-1,0.953 [0.92–0.97],0.955 [0.94–0.97],0.958 [0.92–0.98],0.923 [0.84–0.97]
v3.1 paket,0.881 [0.83–0.91],0.951 [0.93–0.97],0.853 [0.80–0.90],0.774 [0.65–0.86]
D-1 ∨ v3.1,0.888 [0.84–0.92],0.991 [0.99–1.00],0.841 [0.79–0.89],0.767 [0.64–0.85]


,Macro-F1,Se,Sp,PPV
CPSC 2021,,,,
W-F vs D-1,"Δ -0.022 [-0.043…-0.005], P = 0.003 → lebih buruk","Δ +0.038 [+0.024…+0.055], P = 1.000 → lebih baik","Δ -0.053 [-0.075…-0.035], P = 0.000 → lebih buruk","Δ -0.076 [-0.120…-0.045], P = 0.000 → lebih buruk"
W-F vs v3.1,"Δ +0.050 [+0.031…+0.074], P = 1.000 → lebih baik","Δ +0.041 [+0.028…+0.056], P = 1.000 → lebih baik","Δ +0.052 [+0.023…+0.084], P = 1.000 → lebih baik","Δ +0.073 [+0.034…+0.125], P = 1.000 → lebih baik"
D-1 ∨ v3.1 vs D-1,"Δ -0.065 [-0.100…-0.039], P = 0.000 → lebih buruk","Δ +0.037 [+0.023…+0.053], P = 1.000 → lebih baik","Δ -0.117 [-0.154…-0.086], P = 0.000 → lebih buruk","Δ -0.156 [-0.232…-0.100], P = 0.000 → lebih buruk"


,Macro-F1,Se,Sp,PPV
MIT-BIH,,,,
W-F,0.867 [0.72–0.94],0.998 [0.99–1.00],0.929 [0.89–0.96],0.628 [0.31–0.81]
D-1,0.967 [0.90–0.99],0.983 [0.97–1.00],0.987 [0.98–0.99],0.902 [0.69–0.97]
v3.1 paket,0.754 [0.60–0.84],0.916 [0.87–0.96],0.857 [0.79–0.91],0.435 [0.17–0.64]
D-1 ∨ v3.1,0.771 [0.61–0.86],0.997 [0.99–1.00],0.854 [0.79–0.91],0.450 [0.18–0.66]


,Macro-F1,Se,Sp,PPV
MIT-BIH,,,,
W-F vs D-1,"Δ -0.100 [-0.185…-0.047], P = 0.000 → lebih buruk","Δ +0.015 [+0.001…+0.034], P = 0.988 → lebih baik","Δ -0.058 [-0.095…-0.028], P = 0.000 → lebih buruk","Δ -0.274 [-0.405…-0.144], P = 0.000 → lebih buruk"
W-F vs v3.1,"Δ +0.113 [+0.066…+0.165], P = 1.000 → lebih baik","Δ +0.082 [+0.044…+0.127], P = 1.000 → lebih baik","Δ +0.072 [+0.034…+0.118], P = 1.000 → lebih baik","Δ +0.193 [+0.097…+0.288], P = 1.000 → lebih baik"
D-1 ∨ v3.1 vs D-1,"Δ -0.196 [-0.296…-0.116], P = 0.000 → lebih buruk","Δ +0.014 [+0.001…+0.032], P = 0.988 → lebih baik","Δ -0.133 [-0.197…-0.080], P = 0.000 → lebih buruk","Δ -0.452 [-0.571…-0.296], P = 0.000 → lebih buruk"


,Macro-F1,Se,Sp,PPV
LTAFDB,,,,
W-F,0.971 [0.96–0.98],0.989 [0.97–1.00],0.949 [0.92–0.97],0.961 [0.93–0.98]
D-1,0.967 [0.95–0.98],0.974 [0.96–0.99],0.960 [0.93–0.98],0.968 [0.94–0.99]
v3.1 paket,0.933 [0.91–0.95],0.948 [0.93–0.97],0.915 [0.88–0.95],0.933 [0.90–0.96]
D-1 ∨ v3.1,0.944 [0.92–0.96],0.989 [0.97–1.00],0.892 [0.85–0.93],0.920 [0.88–0.95]


,Macro-F1,Se,Sp,PPV
LTAFDB,,,,
W-F vs D-1,"Δ +0.004 [-0.004…+0.012], P = 0.827 → tidak te...","Δ +0.015 [+0.010…+0.021], P = 1.000 → lebih baik","Δ -0.011 [-0.027…+0.006], P = 0.099 → tidak te...","Δ -0.008 [-0.019…+0.005], P = 0.111 → tidak te..."
W-F vs v3.1,"Δ +0.038 [+0.026…+0.052], P = 1.000 → lebih baik","Δ +0.041 [+0.029…+0.053], P = 1.000 → lebih baik","Δ +0.034 [+0.012…+0.061], P = 1.000 → lebih baik","Δ +0.028 [+0.010…+0.051], P = 1.000 → lebih baik"
D-1 ∨ v3.1 vs D-1,"Δ -0.023 [-0.038…-0.010], P = 0.000 → lebih buruk","Δ +0.015 [+0.010…+0.021], P = 1.000 → lebih baik","Δ -0.068 [-0.097…-0.044], P = 0.000 → lebih buruk","Δ -0.049 [-0.077…-0.028], P = 0.000 → lebih buruk"


## 3. Deskriptif: AFDB, PTB-XL, alarm per ritme non-AF

In [4]:
afdb_t, afdb_b = F.afdb(model)
display(pd.DataFrame({k: {m: f"{v[m][0]:.3f} [{v[m][1]:.2f}–{v[m][2]:.2f}]" for m in ("Macro-F1", "Se", "Sp", "PPV")} for k, v in afdb_t.items()}).T.rename_axis("AFDB (5 ulangan)"))
display(pd.DataFrame({k: {m: dl(v) for m, v in b.items()} for k, b in afdb_b.items()}).T.rename_axis("AFDB"))
ptb = F.data_ptbxl()
pred_p = F.prediksi(model, ptb["d1"], ptb["p31"])
b_ptb, t_ptb = F.banding(ptb, pred_p)
ptb_kelas = {k: {"AFIB": float(v[ptb["y"] == 1].mean()), "AFL": float(v[ptb["y"] == 2].mean()), "STACH": float(v[ptb["st"]].mean())} for k, v in pred_p.items()}
display(tabel(t_ptb, "PTB-XL"))
display(tabel_banding(b_ptb, "PTB-XL"))
display(pd.DataFrame(ptb_kelas).T.round(3).rename_axis("PTB-XL: alarm per kelas"))
for nm in ("MIT-BIH", "LTAFDB"):
    display(pd.DataFrame(hasil_h[nm]["ritme"]).T.round(3).rename_axis(f"{nm}: alarm per ritme"))

,Macro-F1,Se,Sp,PPV
AFDB (5 ulangan),,,,
W-F,0.980 [0.97–0.99],0.984 [0.97–0.99],0.978 [0.97–0.99],0.968 [0.95–0.98]
D-1,0.968 [0.95–0.98],0.935 [0.90–0.97],0.992 [0.99–1.00],0.988 [0.98–0.99]
v3.1 paket,0.929 [0.89–0.95],0.939 [0.92–0.96],0.926 [0.85–0.97],0.895 [0.82–0.95]
D-1 ∨ v3.1,0.947 [0.91–0.97],0.985 [0.97–0.99],0.924 [0.85–0.96],0.897 [0.82–0.95]


,Macro-F1,F1 AF1,Se,Sp
AFDB,,,,
W-F vs D-1,"Δ +0.012 [-0.000…+0.025], P = 0.974 → lebih baik","Δ +0.015 [+0.000…+0.032], P = 0.978 → lebih baik","Δ +0.048 [+0.024…+0.077], P = 1.000 → lebih baik","Δ -0.014 [-0.021…-0.008], P = 0.000 → lebih buruk"
W-F vs v3.1,"Δ +0.051 [+0.028…+0.087], P = 1.000 → lebih baik","Δ +0.060 [+0.035…+0.094], P = 1.000 → lebih baik","Δ +0.045 [+0.023…+0.064], P = 1.000 → lebih baik","Δ +0.053 [+0.017…+0.119], P = 1.000 → lebih baik"


,Macro-F1,Se,Sp,PPV
PTB-XL,,,,
W-F,0.906 [0.90–0.91],0.964 [0.95–0.97],0.971 [0.97–0.97],0.725 [0.70–0.74]
D-1,0.939 [0.93–0.94],0.912 [0.90–0.93],0.988 [0.99–0.99],0.862 [0.84–0.88]
v3.1 paket,0.817 [0.81–0.83],0.917 [0.90–0.93],0.935 [0.93–0.94],0.529 [0.51–0.55]
D-1 ∨ v3.1,0.819 [0.81–0.83],0.966 [0.96–0.97],0.929 [0.93–0.93],0.519 [0.50–0.54]


,Macro-F1,Se,Sp,PPV
PTB-XL,,,,
W-F vs D-1,"Δ -0.033 [-0.040…-0.026], P = 0.000 → lebih buruk","Δ +0.052 [+0.040…+0.064], P = 1.000 → lebih baik","Δ -0.018 [-0.020…-0.015], P = 0.000 → lebih buruk","Δ -0.138 [-0.156…-0.121], P = 0.000 → lebih buruk"
W-F vs v3.1,"Δ +0.089 [+0.081…+0.097], P = 1.000 → lebih baik","Δ +0.047 [+0.035…+0.060], P = 1.000 → lebih baik","Δ +0.036 [+0.032…+0.039], P = 1.000 → lebih baik","Δ +0.196 [+0.178…+0.213], P = 1.000 → lebih baik"
D-1 ∨ v3.1 vs D-1,"Δ -0.120 [-0.129…-0.111], P = 0.000 → lebih buruk","Δ +0.054 [+0.042…+0.066], P = 1.000 → lebih baik","Δ -0.059 [-0.063…-0.056], P = 0.000 → lebih buruk","Δ -0.343 [-0.363…-0.325], P = 0.000 → lebih buruk"


,AFIB,AFL,STACH
PTB-XL: alarm per kelas,,,
W-F,0.965,0.94,0.060
D-1,0.926,0.50,0.025
v3.1 paket,0.917,0.92,0.297
D-1 ∨ v3.1,0.967,0.94,0.300


,n,W-F,D-1,v3.1 paket,D-1 ∨ v3.1
MIT-BIH: alarm per ritme,,,,,
(AB,13.0,0.462,0.385,0.462,0.538
(AFIB,1416.0,0.998,0.987,0.916,0.997
(AFL,98.0,1.000,0.939,0.918,1.000
(B,233.0,0.339,0.052,0.571,0.579
(BII,130.0,0.000,0.000,0.000,0.000
(IVR,21.0,0.238,0.524,0.048,0.524
(N,11726.0,0.058,0.009,0.127,0.130
(NOD,5.0,0.400,0.200,0.000,0.200
(PREX,52.0,0.000,0.000,0.058,0.058


,n,W-F,D-1,v3.1 paket,D-1 ∨ v3.1
LTAFDB: alarm per ritme,,,,,
(AB,1503.0,0.340,0.094,0.579,0.616
(AFIB,706582.0,0.989,0.974,0.948,0.989
(B,437.0,0.096,0.023,0.515,0.526
(N,537559.0,0.047,0.028,0.085,0.097
(SBR,25501.0,0.110,0.279,0.042,0.299
(SVTA,599.0,0.259,0.100,0.265,0.265
(T,147.0,0.490,0.238,0.544,0.701


In [5]:
kriteria = dict(
    K1_se_vs_D1=bool(b_ico["W-F vs D-1"]["Se"][3] >= 0.95),
    K2_mf1_vs_D1=bool(b_ico["W-F vs D-1"]["Macro-F1"][3] > 0.05),
    K3_vs_v31=bool(b_ico["W-F vs v3.1"]["Macro-F1"][3] > 0.05 and b_ico["W-F vs v3.1"]["Se"][3] > 0.05),
    K4_sinus_takikardia=bool(lat["W-F"]["alarm"] <= 0.10),
    K5_holter=bool(all(hasil_h[nm]["banding"]["W-F vs D-1"]["Macro-F1"][3] > 0.05 for nm in hasil_h)))
kriteria["W-F_menggantikan_D-1"] = all(kriteria.values())
ring = {"koefisien": koef, "cinc": dict(banding=b_cinc, titik=t_cinc, n=int(len(cinc["y2"]))), "icentia": dict(banding=b_ico, titik=t_ico),
        "icentia_tanpa_cepat": dict(banding=b_ico_tf, titik=t_ico_tf), "latihan": lat, "holter": hasil_h, "afdb": dict(titik=afdb_t, banding=afdb_b),
        "ptbxl": dict(banding=b_ptb, titik=t_ptb, kelas=ptb_kelas), "kriteria": kriteria, "lingkungan": Ev._lingkungan()}
(F.CACHE / "ringkasan.json").write_text(json.dumps(ring, indent=1, default=float, ensure_ascii=False))
print("kriteria:", kriteria)
print("tersimpan:", F.CACHE / "ringkasan.json")

kriteria: {'K1_se_vs_D1': True, 'K2_mf1_vs_D1': True, 'K3_vs_v31': True, 'K4_sinus_takikardia': False, 'K5_holter': False, 'W-F_menggantikan_D-1': False}
tersimpan: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v17/ringkasan.json


## 4. Kesimpulan

Dijalankan penuh 6 November 2026 di PC (±1 menit, semua dari cache). Kriteria yang dikunci: K1, K2, K3 lolos; **K4 dan K5 gagal** → W-F
**tidak menggantikan D-1**. D-1 tetap kandidat detektor AF 2 kelas; paket v3.1 tetap paket produksi.

- **W-F yang dipelajari:** AF bila 0,629·d1 + 0,272·r + 1,616 ≥ 0, yaitu kira-kira d1 > −2,6 − 0,43·r. Dilatih di EKG genggam, ia
  menurunkan ambang ECGFounder dan menambah suara dari model RR.
- **Icentia11k (wearable, uji utama):** sensitivitas 0,921 vs D-1 0,843 dan v3.1 0,871 (lebih baik); Macro-F1 0,913 vs 0,870 dan 0,858
  (lebih baik); spesifisitas 0,934 vs 0,972 (turun). Jadi celah sensitivitas wearable **bisa** ditutup.
- **Harganya (penjaga gagal):** alarm sinus takikardia asli ≥ 120 bpm 56,6% [45–68] (D-1 0,5%; batas 10%), karena suara RR kembali
  mengalarmkan detak cepat yang teratur. CPSC 2021 0,931 vs D-1 0,953 dan MIT-BIH 0,867 vs 0,967 (lebih buruk; spesifisitas turun).
  LTAFDB tidak terbedakan.
- **Pembanding naif D-1 ∨ v3.1:** sensitivitas tertinggi tetapi spesifisitas setara v3.1 dan alarm sinus takikardia 98%.
- **Deskriptif:** AFDB W-F 0,980 vs D-1 0,968; PTB-XL 0,906 vs 0,939 (STACH 6,0% vs 2,5%).
- **Pelajaran:** titik kerja yang baik untuk data wearable (ambang lebih rendah + RR) merusak spesifisitas di Holter dan saat latihan fisik.
  Di CinC 2017 hanya 1,4% window non-AF yang ≥ 120 bpm (563 window), jadi model yang dilatih di sana hampir tidak "melihat" biayanya (pelajaran v1–v6 dan v16 H-X
  berulang). Titik kerja harus dipilih dari data perangkat SIHEDAF yang memuat aktivitas fisik.